In [1]:
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Add the backend folder to sys.path so we can import existing modules
sys.path.append(os.path.abspath('backend'))

from backend.main import _load_dataset, _load_artifact
from backend.models import WINDOW, STRIDE, MODEL_PALETTE

# Configure Seaborn for beautiful matplotlib plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

def plot_models_trajectory(model_names, dataset_filename="v1.csv"):
    print(f"Loading dataset {dataset_filename}...")
    raw, gt_PN, gt_PE, dt_arr = _load_dataset(dataset_filename)

    # Build sliding windows
    print("Building windows...")
    windows = []
    gt_pn = []
    gt_pe = []
    dt_win = []
    
    for i in range(0, len(raw) - WINDOW + 1, STRIDE):
        windows.append(raw[i:i + WINDOW])
        idx = i + WINDOW - 1
        gt_pn.append(gt_PN[idx])
        gt_pe.append(gt_PE[idx])
        dt_win.append(dt_arr[idx])
        
    n = len(windows)
    print(f"Total windows: {n}")
    
    # Setup plot
    fig, ax = plt.subplots()
    
    # Plot Ground Truth
    ax.plot(gt_pe, gt_pn, color='black', linewidth=3, label='Ground Truth')
    
    # Plot start point
    if len(gt_pn) > 0:
        ax.scatter([gt_pe[0]], [gt_pn[0]], color='blue', s=100, zorder=5, label='Start')

    # Run inference and plot for each model
    for idx, model_name in enumerate(model_names):
        print(f"Running inference for {model_name}...")
        try:
            entry = _load_artifact(model_name)
        except Exception as e:
            print(f"Skipping {model_name}: {e}")
            continue
            
        model_nn, artifact = entry["model"], entry["artifact"]
        f_scaler, t_scaler = artifact["f_scaler"], artifact["t_scaler"]
        stride_val = artifact.get("stride", STRIDE)
        
        # Batched inference
        all_pred = []
        batch_sz = 512
        for bi in range(0, n, batch_sz):
            batch_raw = windows[bi:bi + batch_sz]
            batch = np.stack([f_scaler.transform(np.asarray(w, dtype=np.float32)) for w in batch_raw])
            x_t = torch.from_numpy(batch).float()
            
            with torch.no_grad():
                pred_scaled = model_nn(x_t).numpy()
            
            all_pred.append(t_scaler.inverse_transform(pred_scaled))
            
        pred_vel = np.concatenate(all_pred, axis=0)
        
        # GT-anchored integration (reset every 200 windows)
        pred_PN_arr = np.zeros(n)
        pred_PE_arr = np.zeros(n)
        pred_PN_arr[0] = gt_pn[0]
        pred_PE_arr[0] = gt_pe[0]
        
        RESET_INTERVAL = 200
        for t in range(1, n):
            if t % RESET_INTERVAL == 0:
                pred_PN_arr[t] = gt_pn[t]
                pred_PE_arr[t] = gt_pe[t]
            else:
                step_dt = stride_val * dt_win[t]
                pred_PN_arr[t] = pred_PN_arr[t - 1] + pred_vel[t, 0] * step_dt
                pred_PE_arr[t] = pred_PE_arr[t - 1] + pred_vel[t, 1] * step_dt
                
        # Optional: Add offset to y (PN) and x (PE) if overlapping is an issue 
        # offset = idx * 10.0
        # pred_PN_arr += offset
        # pred_PE_arr += offset
        
        color = MODEL_PALETTE.get(model_name, "#888888")
        ax.plot(pred_PE_arr, pred_PN_arr, color=color, linewidth=2, 
                linestyle='--', alpha=0.8, label=model_name)

    # Format the plot
    ax.set_title("AUV Trajectory Tracking: Models vs Ground Truth", fontsize=16, pad=15)
    ax.set_xlabel("East Position (PE) [meters]", fontsize=14)
    ax.set_ylabel("North Position (PN) [meters]", fontsize=14)
    ax.legend(loc='upper right', frameon=True, fancybox=True, shadow=True)
    ax.axis('equal') # Keep the aspect ratio 1:1 so trajectory isn't skewed
    
    plt.tight_layout()
    plt.show()

# Run it
plot_models_trajectory(["RNN", "LSTM", "BERT", "Mamba"], dataset_filename="v1.csv")

ModuleNotFoundError: No module named 'backend'